# Induction circuit analysis for a `replicate.py` run

ARENA 1.2-style analysis of one training run: its config, how the induction and previous-token scores
evolved, attention patterns on repeated random tokens, and the circuit (which heads matter, the
previous-token head's positional attention, OV copying, K-composition).

**Set `RUN` below**, then run all cells. Everything runs on CPU in a minute or two.

How the model is set up for analysis (see `circuits.py`):
- The run's GPT-2 is converted to a TransformerLens `HookedTransformer` and **unrolled**: a looped run
  (1 layer × 2 loops) becomes 2 blocks holding *copies of the same weights*, so block 0 = loop 0 and
  block 1 = loop 1. A 2-layer control is just its 2 layers.
- LayerNorm is **folded** into the neighbouring weights, so each LayerNorm is only "divide by the
  vector's norm". The circuit maths below ignores that per-token scale (as is usual).
- Unlike ARENA's model, these models have MLPs. Wherever ARENA uses the token embedding `W_E`, we also
  show an **effective embedding** `W_E + MLP0(W_E)` (the embedding after block 0's MLP), since that is
  closer to what later heads actually read.

In [ ]:
RUN = "rep_gpt2_L1x2_300M_s3"   # folder name inside checkpoints/
STEP = None                      # None = final checkpoint; or a step listed in phases.json

In [ ]:
from pathlib import Path

import circuitsvis as cv
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import HTML, display
from transformer_lens import FactoredMatrix

from circuits import load_analysis
from replicate import make_ps_tokens

t.set_grad_enabled(False)
a = load_analysis(Path("checkpoints") / RUN, STEP)
model = a.tl
n_blocks, n_heads, d_head = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
tokens_per_step = a.args.batch_size * a.args.ctx
print(f"{RUN}: step {a.step} ({a.step * tokens_per_step / 1e6:.0f}M tokens); blocks: {a.labels}")

# colours: sequential blue for magnitudes, blue-grey-red for signed values, fixed categorical hues
SEQ = [[0, "#ffffff"], [0.05, "#cde2fb"], [0.2, "#86b6ef"], [0.4, "#3987e5"], [0.6, "#256abf"],
       [0.8, "#184f95"], [1, "#0d366b"]]
DIV = [[0, "#184f95"], [0.25, "#6da7ec"], [0.5, "#f0efec"], [0.75, "#e66767"], [1, "#9f2a29"]]
BLUE, ORANGE = "#2a78d6", "#eb6834"


def imshow(z, x=None, y=None, title="", diverging=False, zmin=None, zmax=None, labels=None, height=None, **kw):
    z = t.as_tensor(z).float()
    if diverging:
        lim = zmax if zmax is not None else z.abs().max().item()
        zmin, zmax = -lim, lim
    fig = px.imshow(z.numpy(), x=x, y=y, title=title, color_continuous_scale=DIV if diverging else SEQ,
                    zmin=zmin, zmax=zmax, labels=labels or {}, aspect="auto", **kw)
    fig.update_layout(template="plotly_white", height=height)
    fig.show()


head_names = [f"{b}.H{h}" for b in a.labels for h in range(n_heads)]

## 1. Training run config

In [ ]:
args = vars(a.args)
looped = a.args.n_loops > 1
arch = (f"{a.args.n_layers} layer(s) x {a.args.n_loops} loops (weights shared across loops)" if looped
        else f"{a.args.n_layers} layers, no looping")
unique = {id(p): p for p in a.hf.parameters()}.values()
n_params = sum(p.numel() for p in unique)
n_block = sum(p.numel() for p in {id(p): p for blk in a.hf.transformer.h for p in blk.parameters()}.values())
print(f"architecture: {arch}")
print(f"parameters: {n_params / 1e6:.2f}M total, {n_block / 1e6:.2f}M in the transformer blocks (unique)")
print(f"heads: {n_heads} per block, size {d_head};  MLP width {model.cfg.d_mlp}")
display(pd.DataFrame({"value": {k: v for k, v in args.items()}}))
if a.phases:
    display(pd.DataFrame([{"checkpoint": k, "step": v, "tokens (M)": round(v * tokens_per_step / 1e6, 1),
                           "induction (best head)": next(r["ind_max"] for r in a.metrics if r["step"] >= v)}
                          for k, v in sorted(a.phases.items(), key=lambda kv: kv[1])]))

## 2. Scores over training

Each score is the best head's: **induction** = attention from a token in the second copy of a repeated
random sequence to the token after its first occurrence; **previous-token** = attention to the
immediately preceding token. Dotted lines mark the saved checkpoints.

In [ ]:
m = a.metrics
tok = [r["tokens"] / 1e6 for r in m]
fig = go.Figure()
for kind, key, colour in [("induction", "ind", BLUE), ("previous-token", "prev", ORANGE)]:
    for b, name in enumerate(a.labels):
        fig.add_trace(go.Scatter(x=tok, y=[max(r[key][b]) for r in m], name=f"{kind}, best head in {name}",
                                 line=dict(color=colour, width=2, dash="solid" if b == n_blocks - 1 else "dot")))
for label, s in a.phases.items():
    fig.add_vline(x=s * tokens_per_step / 1e6, line=dict(color="#999", width=1, dash="dot"),
                  annotation_text=label, annotation_textangle=-90, annotation_font_size=10)
fig.update_layout(title=f"{RUN}: best head's score over training", xaxis_title="training tokens (millions)",
                  yaxis_title="score (mean attention)", yaxis_range=[0, 1], hovermode="x unified",
                  template="plotly_white", height=450)
fig.show()

fig = px.line(x=tok, y=[r["val_loss"] for r in m], labels={"x": "training tokens (millions)", "y": "validation loss"},
              title="Validation loss")
fig.update_traces(line=dict(color=BLUE, width=2))
fig.update_layout(template="plotly_white", height=300)
fig.show()

In [ ]:
# every head's score at the end of training (from the last evaluation logged during training)
last = m[-1]
imshow(t.tensor(last["ind"]), x=[f"H{h}" for h in range(n_heads)], y=a.labels, zmin=0, zmax=1,
       title=f"Induction score per head (step {last['step']})", height=250)
imshow(t.tensor(last["prev"]), x=[f"H{h}" for h in range(n_heads)], y=a.labels, zmin=0, zmax=1,
       title=f"Previous-token score per head (step {last['step']})", height=250)

## 3. Attention patterns on repeated random tokens (this checkpoint)

25 random tokens, then the same 25 again (no BOS, as in training). An **induction head** shows a stripe
24 positions below the diagonal in the second half; a **previous-token head** shows the stripe just
below the diagonal.

In [ ]:
HALF = 25
rep = make_ps_tokens(n=1, half=HALF, seed=1)
str_toks = [model.tokenizer.decode([i]) for i in rep[0].tolist()]
logits, cache = model.run_with_cache(rep)
for b, name in enumerate(a.labels):
    display(HTML(f"<h3>{name}</h3>"))
    display(cv.attention.attention_heads(attention=cache["pattern", b][0], tokens=str_toks,
                                         attention_head_names=[f"{name}.H{h}" for h in range(n_heads)]))

In [ ]:
# loss on each token: high in the first copy (random), low in the second copy if induction works
logprobs = logits.log_softmax(-1)[0, :-1].gather(-1, rep[0, 1:, None])[:, 0]
fig = px.line(x=list(range(1, 2 * HALF)), y=(-logprobs).tolist(), markers=True,
              labels={"x": "position of the predicted token", "y": "loss (nats)"},
              title="Per-token loss on the repeated sequence")
fig.add_vline(x=HALF + 0.5, line=dict(color="#999", dash="dot"), annotation_text="second copy starts")
fig.update_traces(line=dict(color=BLUE, width=2), marker=dict(size=8))
fig.update_layout(template="plotly_white", height=350)
fig.show()

## 4. The circuit

### 4.0 Which heads?
Scores on the standard test (100 sequences of 50 random tokens, repeated). By default the
previous-token head is the best one in block 0 and the induction head the best one in the last block;
override `PREV` / `IND` if you want to look at others.

In [ ]:
batch = make_ps_tokens(n=100, half=50)
_, pat_cache = model.run_with_cache(batch, names_filter=lambda n: n.endswith("pattern"), return_type=None)
dest = t.arange(50, 100)
ind_scores = t.stack([pat_cache["pattern", b][:, :, dest, dest - 49].mean((0, 2)) for b in range(n_blocks)])
prev_scores = t.stack([pat_cache["pattern", b][:, :, dest, dest - 1].mean((0, 2)) for b in range(n_blocks)])
del pat_cache

PREV = (0, int(prev_scores[0].argmax()))
IND = (n_blocks - 1, int(ind_scores[-1].argmax()))
name = lambda bh: f"{a.labels[bh[0]]}.H{bh[1]}"
print(f"previous-token head: {name(PREV)} (prev-token score {prev_scores[PREV]:.2f})")
print(f"induction head:      {name(IND)} (induction score {ind_scores[IND]:.2f})")
if looped:
    print(f"(looped run: {name(PREV)} and {name(IND)} share weights with "
          f"{a.labels[1 - PREV[0]]}.H{PREV[1]} and {a.labels[1 - IND[0]]}.H{IND[1]})")

### 4.1 Which heads matter? (ablation)
Zero out one head at a time (in one block only, so for a looped run "loop 0 H7" is removed in loop 0
but still runs in loop 1) and measure how much the loss on the second copy goes up.

In [ ]:
abl = make_ps_tokens(n=20, half=50, seed=2)

def second_half_loss(fwd_hooks=()):
    # unembed only the positions that predict second-copy tokens (full logits would be ~400 MB)
    with model.hooks(fwd_hooks=list(fwd_hooks)):
        _, cc = model.run_with_cache(abl, names_filter="ln_final.hook_normalized", return_type=None)
    lg = cc["ln_final.hook_normalized"][:, 50:-1] @ model.W_U + model.b_U
    return t.nn.functional.cross_entropy(lg.flatten(0, 1), abl[:, 51:].flatten()).item()

base = second_half_loss()
increase = t.zeros(n_blocks, n_heads)
for b in range(n_blocks):
    for h in range(n_heads):
        def zero_head(z, hook, h=h):
            z[:, :, h] = 0
            return z
        increase[b, h] = second_half_loss([(f"blocks.{b}.attn.hook_z", zero_head)]) - base
print(f"second-copy loss with nothing ablated: {base:.2f} nats")
imshow(increase, x=[f"H{h}" for h in range(n_heads)], y=a.labels, diverging=True,
       title="Increase in second-copy loss when one head is zeroed (nats)", height=250)

### 4.2 Previous-token head: does it attend by position? (QK circuit on positional embeddings)
ARENA's previous-token head attends one position back using positional information only. Here:
the attention pattern the head would produce if its queries and keys saw only the positional
embeddings `W_pos` (normalised as LayerNorm would). A clean stripe one below the diagonal means
"previous token" is implemented positionally.

In [ ]:
b, h = PREV
N = 40
pos = model.blocks[b].ln1(model.W_pos[None, :N])[0]          # LayerNorm's normalisation, as the head sees it
q = pos @ model.W_Q[b, h] + model.b_Q[b, h]
k = pos @ model.W_K[b, h] + model.b_K[b, h]
scores = (q @ k.T) / d_head ** 0.5
scores = scores.masked_fill(t.triu(t.ones(N, N, dtype=t.bool), 1), float("-inf"))
pos_pattern = scores.softmax(-1)
imshow(pos_pattern, labels={"x": "key position", "y": "query position"}, zmin=0, zmax=1,
       title=f"{name(PREV)}: attention from positional embeddings alone", height=450)
print(f"fraction of query positions whose top key is the previous position: "
      f"{(pos_pattern[1:].argmax(-1) == t.arange(N - 1)).float().mean():.2f}")

### 4.2b How much of the previous-token head's *actual* attention comes from position?
4.2 is an ablation: it drops the token embedding, so LayerNorm divides the positional embedding by
*its own* norm. Here we instead **split the real attention scores**. Block 0's input at position *p* is
`E_p + P_p` (token + positional embedding). After folding, LayerNorm only divides each position's vector
by one number `s_p` (its norm). Holding each `s_p` at its **actual** value, the score is bilinear in the
two positions' vectors, `B(u, v) = (u / s_p) W_Q W_Kᵀ (v / s_r)ᵀ / √d_head`, so it splits exactly:

| term | formula | varies with |
|---|---|---|
| token × token | `B(E_p, E_r)` | query and key |
| token × position | `B(E_p, P_r)` | query and key |
| position × token | `B(P_p, E_r)` | query and key |
| position × position | `B(P_p, P_r)` | query and key |
| query bias × key token | `b_Q · E_r W_K / s_r` | key only |
| query bias × key position | `b_Q · P_r W_K / s_r` | key only |
| (query side) × key bias, `b_Q · b_K` | | query only: **softmax ignores these** |

Two measures, both exact (each column sums to the real value), over the standard test's second half:
- **excess at the previous position**: how much each term raises the score at *p − 1* above the average
  over all keys *r ≤ p* that the query sees (what softmax responds to);
- **share of the across-key variation**: each term's covariance with the total (centred across keys),
  divided by the total's variance.

Then the attention patterns you get from the real scores, from the position × position terms only, and
from everything *except* those, all with the actual `s_p`.

In [ ]:
assert PREV[0] == 0, "this split assumes the previous-token head is in block 0 (input = token + position)"
b, h = PREV
T = batch.shape[1]
_, c0 = model.run_with_cache(batch, names_filter=lambda n: n in ("blocks.0.ln1.hook_scale", "blocks.0.attn.hook_attn_scores"),
                             stop_at_layer=1)
s = c0["blocks.0.ln1.hook_scale"]                                   # [batch, pos, 1]: LayerNorm's actual divisor
E = model.W_E[batch] / s                                            # token embedding part, as the head sees it
P = model.W_pos[:T] / s                                             # positional part (broadcast over the batch)
WQ, WK, bQ, bK = model.W_Q[b, h], model.W_K[b, h], model.b_Q[b, h], model.b_K[b, h]
bil = lambda q, k: q @ k.transpose(-1, -2) / d_head ** 0.5         # [batch, query, key]
terms = {
    "token x token": bil(E @ WQ, E @ WK),
    "token x position": bil(E @ WQ, P @ WK),
    "position x token": bil(P @ WQ, E @ WK),
    "position x position": bil(P @ WQ, P @ WK),
    "query bias x key token": bil(bQ.expand(T, d_head), E @ WK),
    "query bias x key position": bil(bQ.expand(T, d_head), P @ WK),
}
query_only = ((((E + P) @ WQ) @ bK)[..., None] + bQ @ bK) / d_head ** 0.5   # constant across keys
valid = t.tril(t.ones(T, T, dtype=t.bool))
real = c0["blocks.0.attn.hook_attn_scores"][:, h]
err = (sum(terms.values()) + query_only - real).masked_fill(~valid, 0).abs().max().item()
print(f"terms reproduce the head's actual scores (max error {err:.1e})")

D = t.arange(50, T)                                                 # queries: the second half, as in the score
def centred(x):                                                     # subtract each query's mean over its keys
    x = x.masked_fill(~valid, 0)
    return (x - x.sum(-1, keepdim=True) / valid.sum(-1, keepdim=True)).masked_fill(~valid, 0)
total_c = centred(sum(terms.values()))
var = (total_c[:, D] ** 2).sum() / valid[D].sum() / len(batch)
rows = []
for term, x in terms.items():
    xc = centred(x)
    rows.append({"term": term,
                 "excess at previous position": xc[:, D, D - 1].mean().item(),
                 "share of across-key variation": ((xc[:, D] * total_c[:, D]).sum() / valid[D].sum() / len(batch) / var).item()})
table = pd.DataFrame(rows).set_index("term")
table.loc["total"] = table.sum()
display(table.round(3))
print(f"(real excess at the previous position: {centred(real)[:, D, D - 1].mean():.3f})")

In [ ]:
pos_only = terms["position x position"]
patterns = {"real scores": sum(terms.values()),
            "position x position only": pos_only,
            "everything except position x position": sum(terms.values()) - pos_only}
for label, sc_ in patterns.items():
    pat = sc_.masked_fill(~valid, float("-inf")).softmax(-1)
    print(f"{label:40s}: mean attention to the previous token {pat[:, D, D - 1].mean():.3f}")
    imshow(pat[0, :40, :40], labels={"x": "key position", "y": "query position"}, zmin=0, zmax=1,
           title=f"{name(PREV)}, first 40 positions: pattern from {label}", height=420)

### 4.3 Induction head: does it copy? (OV circuit)
In the second pass (block 1), the residual stream at a position is the sum of: the **token embedding**
`E` and **position embedding** `P` (the "direct stream"), **block 0's attention output**, and **block 0's
MLP output** (computed from the stream after attention). `E` depends only on the token and `P` only on
the position; the attention output, and hence the MLP output, depend on the other tokens in the sequence.
(LayerNorm is ignored in this description; see the note below.)

**Weights only: the token embedding.** The head's effect via `E` alone is `W_E W_OV W_U`: the map
"token attended to" → "contribution to the logits". The simplest copying mechanism would make this
matrix close to diagonal (the logit it raises most is the token attended to), and have it dominate
the head's output from the other components. Top-k accuracy: how often a token's own logit is among
the top k of its row, over 2,000 random tokens (chance ≈ k / 50,257).

In [ ]:
g = t.Generator().manual_seed(0)
sample = t.randperm(model.cfg.d_vocab, generator=g)[:2000]
b, h = IND
W_OV = model.W_V[b, h] @ model.W_O[b, h]
hits = {1: 0, 5: 0}
for i in range(0, len(sample), 250):                             # rows in chunks: 250 x 50,257 at a time
    rows = sample[i:i + 250]
    top5 = (model.W_E[rows] @ W_OV @ model.W_U).topk(5, dim=-1).indices
    hits[1] += (top5[:, 0] == rows).sum().item(); hits[5] += (top5 == rows[:, None]).any(-1).sum().item()
for k in (1, 5):
    print(f"{name(IND)}, W_E W_OV W_U: top-{k} accuracy {hits[k] / len(sample):.3f}")
sub = sample[:50]
imshow(model.W_E[sub] @ W_OV @ model.W_U[:, sub], diverging=True, labels={"x": "logit of token", "y": "token attended to"},
       title=f"{name(IND)}: W_E W_OV W_U on 50 random tokens", height=500)

**Real activations: what would the head copy if it attended only to position *p*?** Take the actual
residual stream vector at a position *p* (input to the induction head's block), apply `W_OV W_U`, and
check whether the largest logit is **the token at *p*** — the token the head would copy if all its
attention went to *p*. Over all positions of the standard repeated-random-token test.

The same check is done for each component of the stream separately (token embedding, position
embedding, the previous-token head, block 0's other heads and bias, block 0's MLP). The last two
columns are additive: how far each component raises *p*'s own token above the average logit, and its
share of the total.

LayerNorm does not affect this test: before `W_OV` it multiplies each position's vector by one positive
number (the same for all of its components), and the final LayerNorm is also one positive number per
position; neither changes which logit is largest, or the shares.

In [ ]:
bi, hi = IND
bp, hp = PREV
assert bp < bi, "the previous-token head must be in an earlier block than the induction head"
t1 = batch[:50]
keep = {"hook_embed", "hook_pos_embed", f"blocks.{bi}.hook_resid_pre"} | \
       {f"blocks.{b}.{x}" for b in range(bi) for x in ("hook_attn_out", "attn.hook_z", "hook_mlp_out")}
_, c = model.run_with_cache(t1, names_filter=lambda n: n in keep, stop_at_layer=bi + 1)
# one head's output is its z vectors times its W_O (TransformerLens's use_attn_result would build a
# batch x pos x head x d_head x d_model temporary, gigabytes even for small batches)
head_out = lambda b, h: c[f"blocks.{b}.attn.hook_z"][:, :, h] @ model.W_O[b, h]
comps = {"token embedding E": c["hook_embed"], "position embedding P": c["hook_pos_embed"].expand_as(c["hook_embed"])}
for b in range(bi):
    attn = c[f"blocks.{b}.hook_attn_out"]
    if b == bp:
        comps[f"{name(PREV)} (previous-token head)"] = head_out(b, hp)
        comps[f"{a.labels[b]} other heads + bias"] = attn - head_out(b, hp)
    else:
        comps[f"{a.labels[b]} attention"] = attn
    comps[f"{a.labels[b]} MLP"] = c[f"blocks.{b}.hook_mlp_out"]
resid = c[f"blocks.{bi}.hook_resid_pre"]
print(f"components sum to the residual stream (max error {(sum(comps.values()) - resid).abs().max():.1e})")
comps = {"full residual stream": resid, **comps}

own = t1                                                          # the token at each position p
W_OVU = model.W_V[bi, hi] @ model.W_O[bi, hi] @ model.W_U          # [d_model, d_vocab], computed once
stats = {k: {"top1": 0., "top5": 0., "ranks": [], "excess": 0.} for k in comps}
for i in range(0, len(t1), 5):                                    # chunks keep memory down
    for k, x in comps.items():
        lg = x[i:i + 5] @ W_OVU                                   # [chunk, pos, d_vocab]
        own_lg = lg.gather(-1, own[i:i + 5, :, None])
        rank = (lg > own_lg).sum(-1)                              # 0 = own token has the largest logit
        stats[k]["top1"] += (rank == 0).float().sum().item()
        stats[k]["top5"] += (rank < 5).float().sum().item()
        stats[k]["ranks"].append(rank.flatten())
        stats[k]["excess"] += (own_lg[..., 0] - lg.mean(-1)).sum().item()
n = t1.numel()
total_excess = stats["full residual stream"]["excess"]
table = pd.DataFrame({k: {"top-1: own token wins": v["top1"] / n, "top-5": v["top5"] / n,
                          "median rank of own token": t.cat(v["ranks"]).float().median().item(),
                          "own-token logit above mean": v["excess"] / n,
                          "share of the total": v["excess"] / total_excess if k != "full residual stream" else 1.0}
                      for k, v in stats.items()}).T
display(table.round(3))
print(f"chance top-1: {1 / model.cfg.d_vocab:.1e}; the shares of the components sum to "
      f"{table['share of the total'].iloc[1:].sum():.3f}")
del c, comps, lg, W_OVU

### 4.4 K-composition: does the induction head find its target through the previous-token head?
ARENA's test. The induction head's **query** is the current token *A*; its **key** at a position
should encode "the previous token was *A*", which only the previous-token head can supply. So compare
query token *i* against keys built from the previous-token head's output for token *j*:

`(E_i W_Q^ind) · (E_j W_OV^prev W_K^ind)` — a diagonal means K-composition.

In [ ]:
def effective_embed(ids):
    # token embedding after block 0's MLP (attention ignored); not a vector the model computes in context
    x = model.W_E[ids][None]
    return (x + model.blocks[0].mlp(model.blocks[0].ln2(x)))[0]

bp, hp = PREV
bi, hi = IND
sub = sample[:1000]
E = effective_embed(sub)
# LayerNorm at the induction head divides the whole residual stream by its norm; use the typical value
_, sc = model.run_with_cache(batch[:20], names_filter=lambda n: n.endswith("ln1.hook_scale"), stop_at_layer=bi + 1)
s = sc[f"blocks.{bi}.ln1.hook_scale"].mean()
q = (E / s) @ model.W_Q[bi, hi]                                                            # query: current token
prev_out = model.blocks[bp].ln1(E[None])[0] @ model.W_V[bp, hp] @ model.W_O[bp, hp]        # what PREV writes
k = (prev_out / s) @ model.W_K[bi, hi]                                                     # key: read from PREV's output
kcomp = q @ k.T / d_head ** 0.5
acc = (kcomp.argmax(-1) == t.arange(len(sub))).float().mean()
print(f"K-composition circuit {name(PREV)} -> {name(IND)}: top-1 accuracy {acc:.3f} over {len(sub)} tokens "
      f"(chance {1 / len(sub):.3f})")
imshow(kcomp[:50, :50], diverging=True, labels={"x": "previous token of the key", "y": "query token"},
       title="K-composition circuit (50 tokens)", height=500)

### 4.5 Composition scores
How strongly each head's output is read by each later head, through its query (Q), key (K) or value
(V): `‖W_A W_B‖ / (‖W_A‖ ‖W_B‖)` with `W_A` the earlier head's OV matrix. The grey line in the
summary is the average score for random matrices of the same shape. These only see direct
head-to-head paths, not paths through block 0's MLP.

In [ ]:
def random_comp_score(n=100):
    d = model.cfg.d_model
    vals = []
    for _ in range(n):
        A = FactoredMatrix(t.randn(d, d_head), t.randn(d_head, d))
        B = FactoredMatrix(t.randn(d, d_head), t.randn(d_head, d))
        vals.append(((A @ B).norm() / (A.norm() * B.norm())).item())
    return sum(vals) / n

baseline = random_comp_score()
for mode in ["Q", "K", "V"]:
    scores = model.all_composition_scores(mode)          # [layer, head, layer, head]
    sc = scores[0, :, n_blocks - 1, :]                   # block 0 heads -> last block heads
    imshow(sc, x=[f"{a.labels[-1]}.H{h}" for h in range(n_heads)], y=[f"{a.labels[0]}.H{h}" for h in range(n_heads)],
           zmin=0, title=f"{mode}-composition scores (random baseline {baseline:.3f})", height=420)
    print(f"{mode}: {name(PREV)} -> {name(IND)} = {sc[PREV[1], IND[1]]:.3f}  (max {sc.max():.3f}, baseline {baseline:.3f})")

### 4.6 Where does the induction head's attention score come from? (decomposing queries and keys)
On the standard test, split the residual stream that feeds the induction head into its parts
(embedding, positional embedding, each earlier head, earlier MLPs, biases), push each part through the
head's query and key maps, and see which (query part, key part) pairs make up the attention score at
the induction positions. If the head works by K-composition, the big entry is
**query = embedding (or MLP), key = the previous-token head**.

These models have an MLP between the two attention layers, and the previous-token head's output can
reach the induction head *through* it. So block 0's MLP output is split in two: what it would be with
the previous-token head removed, and the change the previous-token head causes. K-composition via the
MLP shows up as **key = "MLP: change from" the previous-token head**.

The numbers are each pair's contribution to *(score at the induction position) − (average score over
all keys that query can see)*: softmax ignores anything that raises every key equally, so this is the
part of the score that actually steers attention. The contributions add up exactly to that excess.

In [ ]:
dec = make_ps_tokens(n=16, half=50, seed=3)
keep = {"hook_embed", "hook_pos_embed", f"blocks.{bi}.hook_resid_pre", f"blocks.{bi}.ln1.hook_scale",
        f"blocks.{bi}.attn.hook_attn_scores"} | {f"blocks.{b}.{x}" for b in range(bi) for x in ("attn.hook_z", "hook_mlp_out")}
_, c = model.run_with_cache(dec, names_filter=lambda n: n in keep, stop_at_layer=bi + 1)
bi, hi = IND
parts, names_ = [c["hook_embed"], c["hook_pos_embed"].expand_as(c["hook_embed"])], ["embed", "pos"]
for b in range(bi):
    for h in range(n_heads):
        parts.append(c[f"blocks.{b}.attn.hook_z"][:, :, h] @ model.W_O[b, h]); names_.append(f"{a.labels[b]}.H{h}")
    parts.append(model.b_O[b].expand_as(c["hook_embed"]).clone()); names_.append(f"{a.labels[b]} attn bias")
    if b == PREV[0]:
        # the MLP can pass the previous-token head's output on; split its output into what it would
        # be without that head, and the change the head causes (exact split, since they add up)
        def drop_prev(z, hook):
            z[:, :, PREV[1]] = 0
            return z
        with model.hooks(fwd_hooks=[(f"blocks.{b}.attn.hook_z", drop_prev)]):
            _, c_abl = model.run_with_cache(dec, names_filter=f"blocks.{b}.hook_mlp_out", stop_at_layer=b + 1)
        mlp_without = c_abl[f"blocks.{b}.hook_mlp_out"]
        parts += [mlp_without, c[f"blocks.{b}.hook_mlp_out"] - mlp_without]
        names_ += [f"{a.labels[b]} MLP (no {name(PREV)})", f"{a.labels[b]} MLP: change from {name(PREV)}"]
    else:
        parts.append(c[f"blocks.{b}.hook_mlp_out"]); names_.append(f"{a.labels[b]} MLP")
parts = t.stack(parts)                                                      # [part, batch, pos, d_model]
err = (parts.sum(0) - c[f"blocks.{bi}.hook_resid_pre"]).abs().max().item()
print(f"parts sum to the residual stream (max error {err:.1e})")

scale = c[f"blocks.{bi}.ln1.hook_scale"]                                    # LayerNorm's per-token divisor
qp = (parts / scale) @ model.W_Q[bi, hi]
kp = (parts / scale) @ model.W_K[bi, hi]
# the query and key biases are one more part each
qp = t.cat([qp, model.b_Q[bi, hi].expand_as(qp[:1])]); kp = t.cat([kp, model.b_K[bi, hi].expand_as(kp[:1])])
names_ = names_ + ["q/k bias"]
dest = t.arange(50, 100); src = dest - 49
# softmax only cares how much a key's score exceeds the other keys' scores for the same query, so
# measure each pair's contribution to (score at the induction position) - (mean score over keys 0..dest)
k_cummean = kp.cumsum(2) / t.arange(1, kp.shape[2] + 1)[:, None]             # mean key over positions 0..p
qs = qp[:, :, dest]                                                          # [part, batch, n_dest, d_head]
ks = kp[:, :, src] - k_cummean[:, :, dest]
contrib = t.einsum("ibnd,jbnd->ij", qs, ks) / (qs.shape[1] * qs.shape[2]) / d_head ** 0.5
scores_all = c[f"blocks.{bi}.attn.hook_attn_scores"][:, hi]                  # [batch, dest, src], masked = -inf
masked = scores_all.masked_fill(t.triu(t.ones(100, 100, dtype=t.bool), 1), 0)
excess = (scores_all[:, dest, src] - masked[:, dest].sum(-1) / (dest + 1)).mean().item()
print(f"induction-position score minus mean score over the query's keys: {excess:.3f}; "
      f"sum of all part pairs: {contrib.sum():.3f}")

top = contrib.flatten().abs().topk(6).indices
for i in top.tolist():
    qi, kj = divmod(i, len(names_))
    print(f"  query {names_[qi]:>16s}  x  key {names_[kj]:<16s}  {contrib[qi, kj]:+.2f}")
del c, parts, qp, kp, qs, ks
keep = contrib.abs().amax(0).topk(min(8, len(names_))).indices.sort().values   # the key parts that matter most
imshow(contrib[:, keep], x=[names_[i] for i in keep], y=names_, diverging=True,
       labels={"x": "key part", "y": "query part"},
       title=f"{name(IND)}: how much each (query part, key part) raises the induction position above the average key", height=550)